# 05. Machine Learning Demand Forecasting & Evaluation

## Methodological Workflow
1. Feature Engineering: Lags (1, 7, 14, 28), Rolling statistics (7, 14, 28), Calendar seasonality.
2. Chronological Train/Test Split (Jan 2024–Mar 2025 Train, Apr–Jun 2025 Test Holdout).
3. Random Forest Regression demand modeling.
4. Interview-grade evaluation: MAE, RMSE, MAPE, R2 score.
5. Integration into Multi-Factor Inventory Risk Engine.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
import sys

sys.path.append(os.path.abspath('..'))
from src.forecasting.feature_engineering import build_demand_features, split_chronological
from src.forecasting.train_model import train_demand_model, get_feature_importance
from src.forecasting.evaluate_model import evaluate_forecasts

PROCESSED_DIR = os.path.join('..', 'data', 'processed')
fact_sales = pd.read_csv(os.path.join(PROCESSED_DIR, 'fact_sales.csv'))

df_features = build_demand_features(fact_sales)
X_train, y_train, X_test, y_test, train_df, test_df, feature_cols = split_chronological(df_features, split_date='2025-04-01')

model = train_demand_model(X_train, y_train)
y_pred = model.predict(X_test)
metrics = evaluate_forecasts(y_test, y_pred)

### Feature Importance Weights

In [ ]:
feat_imp = get_feature_importance(model, feature_cols)

plt.figure(figsize=(10, 5))
plt.barh(feat_imp['feature'][::-1], feat_imp['importance_pct'][::-1], color='#6366F1')
plt.title('Demand Forecasting - Feature Importance (%)', fontsize=14, fontweight='bold')
plt.xlabel('Importance Percentage (%)')
plt.show()

### Actual vs Forecast Demand Comparison (Test Sample)

In [ ]:
test_sample = test_df.copy()
test_sample['predicted_demand'] = y_pred

sample_series = test_sample[(test_sample['product_id'] == test_sample['product_id'].iloc[0]) & 
                            (test_sample['warehouse_id'] == 'W001')].sort_values('sale_date')

plt.figure(figsize=(12, 5))
plt.plot(pd.to_datetime(sample_series['sale_date']), sample_series['quantity_sold'], label='Actual Demand', color='#1E293B', linewidth=2)
plt.plot(pd.to_datetime(sample_series['sale_date']), sample_series['predicted_demand'], label='Random Forest Forecast', color='#38BDF8', linestyle='--', linewidth=2)
plt.title(f"Actual vs Predicted Daily Demand ({sample_series['product_id'].iloc[0]} @ W001)", fontsize=14, fontweight='bold')
plt.xlabel('Date')
plt.ylabel('Units Sold / Day')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()